In [ ]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from paths import PROCESSED, RESULTS, savefig
from pypsa_helpers import solve_scenario, require_optimal, total_co2, total_system_cost, scale_capital_cost

import pypsa
import pandas as pd
import matplotlib.pyplot as plt

Sensitivity: Electrolysis Capital Cost (100% CO2 Reduction Scenario)

Per the assignment: for the zero-CO2 scenario, successively reduce one technology's
capital cost down to 0% in 25% steps (100/75/50/25/0%) and see how the optimised system
changes. Electrolysis (the H2 electrolyzer) was chosen.

This notebook is parameterized by the `ELECTROLYSIS_COST_FRACTION` environment variable
(defaults to `1.0`, i.e. baseline cost) so the same notebook file gets run 5 times - once
per cost level - rather than duplicating it five times. Each run solves independently and
saves its own result file; `10_sensitivity_electrolysis_summary.ipynb` then loads all five
results together for comparison.

In [ ]:
cost_fraction = float(os.environ.get("ELECTROLYSIS_COST_FRACTION", "1.0"))
assert 0.0 <= cost_fraction <= 1.0
print("electrolysis capital cost fraction:", cost_fraction)

In [ ]:
n = pypsa.Network(str(PROCESSED / "pypsa_network_base.nc"))
cost_scale = n.meta["cost_scale"]

n = scale_capital_cost(n, carrier="H2 electrolysis", fraction=cost_fraction)
n = solve_scenario(n, co2_limit=0)

In [ ]:
print("status:", n.meta["status"], "| termination condition:", n.meta["termination_condition"])

# Refuse to report numbers from a solve that didn't actually finish (see 07/08 - HiGHS's
# objective reads back as 0.0 before crossover completes).
require_optimal(n)

total_cost_eur = total_system_cost(n, cost_scale=cost_scale)
total_co2_t = total_co2(n)
electrolysis_capacity_mw = n.links.loc[n.links.carrier == "H2 electrolysis", "p_nom_opt"].sum()

print(f"electrolysis cost fraction:     {cost_fraction:.0%}")
print(f"total annual system cost:       {total_cost_eur:,.0f} EUR/yr")
print(f"total CO2 emissions:            {total_co2_t:,.0f} t/yr")
print(f"electrolysis capacity built:    {electrolysis_capacity_mw:,.0f} MW")

In [ ]:
pct_label = f"{round(cost_fraction * 100)}pct"
n.export_to_netcdf(str(PROCESSED / f"pypsa_sensitivity_electrolysis_{pct_label}.nc"))